#### 1142_DS_Lab3  資料聚合及遮罩應用於股價分析

# Aggregations(聚合函數): Min, Max, and Everything In Between
Often when faced with a large amount of data, a first step is to compute summary statistics for the data in question. Perhaps the most common summary statistics are the mean and standard deviation, which allow you to summarize the "typical" values in a dataset, but other aggregates are useful as well (the sum, product, median, minimum and maximum, quantiles, etc.).
** Most texts are released under the [CC-BY-NC-ND license](https://creativecommons.org/licenses/by-nc-nd/3.0/us/legalcode), and code is released under the [MIT license](https://opensource.org/licenses/MIT).

In [3]:
import numpy as np
A = np.random.random(100)
A

array([0.09104337, 0.81469292, 0.07604162, 0.04299678, 0.542085  ,
       0.96829841, 0.94710237, 0.78162712, 0.15266314, 0.61750363,
       0.82760982, 0.59956186, 0.27076937, 0.14389814, 0.09201695,
       0.62341406, 0.84910781, 0.69586561, 0.9848741 , 0.8931242 ,
       0.13305996, 0.5797796 , 0.84541546, 0.54174668, 0.81410051,
       0.93292211, 0.86230769, 0.84933805, 0.51935984, 0.08777341,
       0.68044416, 0.71005612, 0.66327833, 0.25593333, 0.2410765 ,
       0.9911952 , 0.06399137, 0.43722289, 0.07521437, 0.59195125,
       0.48267841, 0.28191519, 0.58574308, 0.39127242, 0.89691945,
       0.17339139, 0.66698133, 0.68912408, 0.17690168, 0.88688925,
       0.02654619, 0.70373235, 0.64846676, 0.1608994 , 0.48888872,
       0.42917846, 0.32226254, 0.52296824, 0.63692434, 0.69818912,
       0.50910557, 0.88932863, 0.47031093, 0.93385223, 0.30325749,
       0.5071576 , 0.00267319, 0.85909161, 0.68878885, 0.52547373,
       0.78551387, 0.66218298, 0.78008227, 0.79481377, 0.72395

In [4]:
np.sum(A)
%timeit sum(A)
%timeit np.sum(A)

5 μs ± 25.6 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
1.55 μs ± 27.9 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [5]:
big_array = np.random.rand(1000)
%timeit sum(big_array)
%timeit np.sum(big_array)

48.8 μs ± 168 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
1.74 μs ± 15.3 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [6]:
np.min(big_array), np.max(big_array)  # 比較快

(np.float64(0.0016257124400896883), np.float64(0.9975490998712383))

In [7]:
M = np.random.random((3, 4))
print(M)

[[0.16128821 0.89139354 0.06670458 0.59658634]
 [0.40295883 0.6688653  0.05330373 0.81003702]
 [0.33289223 0.10835595 0.53170661 0.0618097 ]]


In [8]:
print(sum(M))

[0.89713927 1.66861478 0.65171493 1.46843306]


In [9]:
print(sum(sum(M)))

4.685902036547178


In [10]:
M.sum()

np.float64(4.685902036547178)

## Aggregation functions take an additional argument specifying the axis along which the aggregate is computed.
For example, we can find the minimum value within each column by specifying axis=0. The axis keyword specifies the dimension of the array that will be collapsed, rather than the dimension that will be returned. So specifying axis=0 means that the first axis will be collapsed: for two-dimensional arrays, this means that values within each column will be aggregated.

![AXIS範例](axis.jpg)

In [11]:
M.sum(axis=0)

array([0.89713927, 1.66861478, 0.65171493, 1.46843306])

In [13]:
M.sum(axis=1)

array([1.71597267, 1.93516487, 1.03476449])

The following table provides a list of useful aggregation functions available in NumPy:

|Function Name      |   NaN-safe Version  | Description                                   |
|-------------------|---------------------|-----------------------------------------------|
| ``np.sum``        | ``np.nansum``       | Compute sum of elements                       |
| ``np.prod``       | ``np.nanprod``      | Compute product of elements                   |
| ``np.mean``       | ``np.nanmean``      | Compute mean of elements                      |
| ``np.std``        | ``np.nanstd``       | Compute standard deviation                    |
| ``np.var``        | ``np.nanvar``       | Compute variance                              |
| ``np.min``        | ``np.nanmin``       | Find minimum value                            |
| ``np.max``        | ``np.nanmax``       | Find maximum value                            |
| ``np.argmin``     | ``np.nanargmin``    | Find index of minimum value                   |
| ``np.argmax``     | ``np.nanargmax``    | Find index of maximum value                   |
| ``np.median``     | ``np.nanmedian``    | Compute median of elements                    |
| ``np.percentile`` | ``np.nanpercentile``| Compute rank-based statistics of elements     |
| ``np.any``        | N/A                 | Evaluate whether any elements are true        |
| ``np.all``        | N/A                 | Evaluate whether all elements are true        |



In [15]:
import pandas as pd
import numpy as np
import requests

In [16]:
date = "20260316"
url = f'https://www.twse.com.tw/exchangeReport/MI_INDEX?response=json&date={date}&type=ALLBUT0999'
response = requests.get(url, verify=False)  # 忽略 SSL 證書驗證
response_json = response.json()

# 找到每日收盤行情的數據
stock_data = []
stock_fields = []
for table in response_json['tables']:
    if "每日收盤行情" in table['title']:  # 確保是股票成交資訊
        stock_data = table['data']  # 股票成交數據
        stock_fields = table['fields']  # 欄位名稱
        break

# 確保有數據才建立 DataFrame
if stock_data and stock_fields:
    stock = pd.DataFrame(stock_data, columns=stock_fields)
else:
    print("Error: 未找到每日收盤行情的數據")

origin = stock_data.copy()

stock.head()

/home/yfhd/Documents/114-2/Data Science/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.twse.com.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,證券代號,證券名稱,成交股數,成交筆數,成交金額,開盤價,最高價,最低價,收盤價,漲跌(+/-),漲跌價差,最後揭示買價,最後揭示買量,最後揭示賣價,最後揭示賣量,本益比
0,0050,元大台灣50,"105,392,137","135,317","7,994,232,270",76.30,76.70,75.45,75.60,<p style= color:green>-</p>,0.35,75.55,"2,075",75.60,69,0.00
1,0051,元大中型100,"151,898","1,061","16,590,643",109.20,110.00,108.25,109.75,<p style= color:red>+</p>,1.15,109.50,2,109.80,1,0.00
2,0052,富邦科技,"40,639,135","24,138","1,837,960,832",45.50,45.72,44.96,45.17,<p style= color:green>-</p>,0.13,45.16,492,45.17,384,0.00
3,0053,元大電子,"18,326",196,"3,091,634",169.60,169.60,167.90,169.05,<p style= color:green>-</p>,0.55,167.90,1,168.75,1,0.00
4,0055,元大MSCI金融,"234,868",566,"7,644,018",32.35,32.69,32.35,32.62,<p style= color:red>+</p>,0.27,32.61,1,32.62,2,0.00


In [17]:
dayprice = np.array(stock['收盤價'])   #也可以這樣寫  dayprice = np.array(stock.收盤價)
print(dayprice)

['75.60' '109.75' '45.17' ... '18.45' '34.90' '129.50']


In [18]:
print(len(dayprice))
print(type(dayprice))

1344
<class 'numpy.ndarray'>


In [19]:
a = np.array(stock.收盤價[0:9])
print(a)

['75.60' '109.75' '45.17' '169.05' '32.62' '38.81' '224.40' '23.86'
 '137.35']


In [21]:
print(len(a))
print(type(a))

9
<class 'numpy.ndarray'>


In [22]:
a= a.astype(float)
print(a)

[ 75.6  109.75  45.17 169.05  32.62  38.81 224.4   23.86 137.35]


In [23]:
print(a.mean())

95.17888888888889


In [ ]:
print("Mean 收盤價:", a.mean())
print("Standard 收盤價:", a.std())
print("Minimum 收盤價:    ", a.min())
print("Maximum 收盤價:    ", a.max())
print("25th percentile:   ", np.percentile(a, 25))
print("Median:            ",  np.median(a))
print("75th percentile:   ", np.percentile(a, 75))

Mean 收盤價: 95.17888888888889
Standard 收盤價: 66.00288587031537
Minimum 收盤價:     23.86
Maximum 收盤價:     224.4
25th percentile:    38.81
Median:             75.6
75th percentile:    137.35


In [ ]:
print("Mean 收盤價:", dayprice.mean())
print("Standard 收盤價:", dayprice.std())
print("Minimum 收盤價:    ", dayprice.min())
print("Maximum 收盤價:    ", dayprice.max())
print("25th percentile:   ", dayprice.percentile(heights, 25))
print("Median:            ", dayprice.median(heights))
print("75th percentile:   ", dayprice.percentile(heights, 75))

<class 'str'>


TypeError: ufunc 'divide' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [28]:
print(dayprice[0:40])

['75.60' '109.75' '45.17' '169.05' '32.62' '38.81' '224.40' '23.86'
 '137.35' '171.60' '40.98' '37.34' '32.03' '175.50' '--' '463.60' '14.23'
 '47.42' '3.27' '53.00' '27.16' '8.51' '19.61' '7.06' '15.98' '84.65'
 '4.31' '26.57' '18.61' '--' '50.25' '67.25' '109.65' '4.50' '17.68'
 '5.30' '33.18' '44.41' '7.16' '32.95']


In [29]:
b= np.array(stock.收盤價[0:40])

In [30]:
print(b)
print(b.size)

['75.60' '109.75' '45.17' '169.05' '32.62' '38.81' '224.40' '23.86'
 '137.35' '171.60' '40.98' '37.34' '32.03' '175.50' '--' '463.60' '14.23'
 '47.42' '3.27' '53.00' '27.16' '8.51' '19.61' '7.06' '15.98' '84.65'
 '4.31' '26.57' '18.61' '--' '50.25' '67.25' '109.65' '4.50' '17.68'
 '5.30' '33.18' '44.41' '7.16' '32.95']
40


In [31]:
b= b.astype(float)

ValueError: could not convert string to float: '--'

In [32]:
b_new=(b!='--')
print(b_new)

[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True False  True  True  True  True  True  True  True  True  True
  True  True  True  True  True False  True  True  True  True  True  True
  True  True  True  True]


In [33]:
b1= b[b_new].astype(float)

In [34]:
print(b1)
print(b1.size)

[ 75.6  109.75  45.17 169.05  32.62  38.81 224.4   23.86 137.35 171.6
  40.98  37.34  32.03 175.5  463.6   14.23  47.42   3.27  53.    27.16
   8.51  19.61   7.06  15.98  84.65   4.31  26.57  18.61  50.25  67.25
 109.65   4.5   17.68   5.3   33.18  44.41   7.16  32.95]
38


# Comparisons, Masks, and Boolean Logic

## Comparison Operators as ufuncs
The result of these comparison operators is always an array with a Boolean data type. All six of the standard comparison operations are available:

In [36]:
x = np.array([1, 2, 3, 4, 5])

In [141]:
x < 3  # less than

array([ True,  True, False, False, False])

In [142]:
x > 3  # greater than

array([False, False, False,  True,  True])

In [143]:
x <= 3  # less than or equal

array([ True,  True,  True, False, False])

In [144]:
x >= 3  # greater than or equal

array([False, False,  True,  True,  True])

In [145]:
x != 3  # not equal

array([ True,  True, False,  True,  True])

In [146]:
x == 3  # equal

array([False, False,  True, False, False])

In [37]:
(2 * x) == (x ** 2)

array([False,  True, False, False, False])

## Working with Boolean Arrays

Given a Boolean array, there are a host of useful operations you can do.
We'll work with ``x``, the two-dimensional array we created earlier.

In [148]:
x=np.arange(12).reshape(3,4)
print(x)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]


In [149]:
# how many values less than 6?
np.count_nonzero(x < 6)

6

In [150]:
# how many values less than 6 in each row?
np.sum(x<6 , axis=1)

array([4, 2, 0])

In [151]:
# are there any values greater than 8?
np.any(x > 8)

np.True_

In [152]:
# are all values less than 12?
np.all(x < 12)

np.True_

In [153]:
np.where(x==7)

(array([1]), array([3]))

## 當日股價分析

In [38]:
dayprice = np.array(stock.收盤價)

In [39]:
dayprice_yes=(dayprice!='--')
print(dayprice_yes[0:40])

[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True False  True  True  True  True  True  True  True  True  True
  True  True  True  True  True False  True  True  True  True  True  True
  True  True  True  True]


In [40]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,015.00'

In [41]:
np.where(dayprice=='1,020.00')

(array([], dtype=int64),)

In [158]:
dayprice[379]

'1,020.00'

In [43]:
dayprice[379]='1020'

In [160]:
dayprice[379]

'1020'

In [42]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,015.00'

In [162]:
np.where(dayprice=='3,365.00')

(array([471]),)

In [163]:
dayprice[471]='3365'

In [164]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,385.00'

In [165]:
np.where(dayprice=='1,385.00')

(array([508]),)

In [166]:
dayprice[508]='1385'

In [167]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,865.00'

In [168]:
np.where(dayprice=='1,865.00')

(array([520, 780]),)

In [169]:
dayprice[520]='1865'
dayprice[780]='1865'

In [170]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,420.00'

In [171]:
np.where(dayprice=='1,420.00')

(array([528]),)

In [172]:
dayprice[528]='1420'

In [173]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,530.00'

In [174]:
np.where(dayprice=='1,530.00')

(array([541]),)

In [175]:
dayprice[541]='1530'

In [176]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '2,705.00'

In [177]:
np.where(dayprice=='2,705.00')

(array([558]),)

In [178]:
dayprice[558]='2705'

In [179]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,010.00'

In [180]:
np.where(dayprice=='1,010.00')

(array([570]),)

In [181]:
dayprice[570]='1010'

In [182]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,720.00'

In [183]:
np.where(dayprice=='1,720.00')

(array([606]),)

In [184]:
dayprice[606]='1720'

In [185]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '2,345.00'

In [186]:
np.where(dayprice=='2,345.00')

(array([773]),)

In [187]:
dayprice[773]='2345'

In [188]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '2,285.00'

In [189]:
np.where(dayprice=='2,285.00')

(array([856]),)

In [190]:
dayprice[856]='2285'

In [191]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,750.00'

In [192]:
np.where(dayprice=='1,750.00')

(array([869]),)

In [193]:
dayprice[869]='1750'

In [194]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '3,245.00'

In [195]:
np.where(dayprice=='3,245.00')

(array([889]),)

In [196]:
dayprice[889]='3245'

In [197]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '3,190.00'

In [198]:
np.where(dayprice=='3,190.00')

(array([890]),)

In [199]:
dayprice[890]='3190'

In [200]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,590.00'

In [201]:
np.where(dayprice=='1,590.00')

(array([891]),)

In [202]:
dayprice[891]='1590'

In [203]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,225.00'

In [204]:
np.where(dayprice=='1,225.00')

(array([998]),)

In [207]:
dayprice[998]='1225'

In [208]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,690.00'

In [209]:
np.where(dayprice=='1,690.00')

(array([1094]),)

In [210]:
dayprice[1094]='1690'

In [211]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '6,285.00'

In [212]:
np.where(dayprice=='6,285.00')

(array([1106]),)

In [213]:
dayprice[1106]='6285'

In [214]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '4,070.00'

In [215]:
np.where(dayprice=='4,070.00')

(array([1138]),)

In [216]:
dayprice[1138]='4070'

In [217]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,005.00'

In [218]:
np.where(dayprice=='1,005.00')

(array([1161]),)

In [219]:
dayprice[1161]='1005'

In [220]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,755.00'

In [221]:
np.where(dayprice=='1,755.00')

(array([1169]),)

In [222]:
dayprice[1169]='1755'

In [223]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '1,320.00'

In [224]:
np.where(dayprice=='1,320.00')

(array([1227]),)

In [225]:
dayprice[1227]='1320'

In [226]:
dayprice= dayprice[dayprice_yes].astype(float)

ValueError: could not convert string to float: '3,950.00'

In [227]:
np.where(dayprice=='3,950.00')

(array([1229]),)

In [228]:
dayprice[1229]='3950'

In [46]:
dayprice= dayprice[dayprice_yes].astype(float)

In [47]:
import pandas as pd
import numpy as np

# 將 dayprice 中的逗號移除並轉換為數字
dayprice = pd.to_numeric(
    pd.Series(dayprice).astype(str).str.replace(',', '', regex=False),
    errors='coerce'
).to_numpy()

In [48]:
print("Mean 當日收盤價:", dayprice.mean())
print("Standard 當日收盤價:", dayprice.std())
print("Minimum 當日收盤價:    ", dayprice.min())
print("Maximum 當日收盤價:    ", dayprice.max())
print("25th percentile:   ", np.percentile(dayprice, 25))
print("Median:            ", np.median(dayprice))
print("75th percentile:   ", np.percentile(dayprice, 75))

Mean 當日收盤價: 108.50113943028487
Standard 當日收盤價: 344.1352816775017
Minimum 當日收盤價:     0.73
Maximum 當日收盤價:     6160.0
25th percentile:    18.8
Median:             36.4
75th percentile:    77.775


# 作業二(Due 2026/03/31): 
請用 "date = "2026/03/17" 分析股市的開盤價和收盤價的平均值和標準差。